In [1]:
# run first
%load_ext autoreload
%autoreload 2

#### Imports

In [2]:
import os
import cupy as cp
import numpy as np
import matplotlib.pyplot as plt

from aicspylibczi import CziFile
from PIL import Image, ImageDraw, ImageFont
from airy_pipeline.methods import to_directory, read_czi, read_yaml, read_beads, read_h5, read_tiff, \
     read_shape, read_kernel, to_mip, to_shift, configure_plots, to_shifted, to_ellipsoid, \
     make_color, to_fmt, segmentation, detection, finding, deskewing, deconvolution

#### Configure

In [3]:
def labels_proc(frame, frame_path, c, lookup):
    """
    Process job for MIP-ing a frame with 
    nuclei labels.

    Parameters
    ----------
    frame : int
        Index of the movie frames.
    directory : str
        Path to the directory for outputs.
    c : dict
        Dictionary for config.
    lookup : dict
        Dictionary of nuclei tracks.
    """
    # get frame path and read info
    info = np.loadtxt(os.path.join(frame_path, "info.csv"),
                      delimiter=",", skiprows=1, ndmin=2)
    # get deconvolve
    deconvolve = read_tiff(os.path.join(frame_path, "deconvolve.tiff"))

    # make MIP
    chnl = c["segmentation"]["channel"]
    mip = to_mip(deconvolve[chnl], gray=True, axis=0, **c["movies"]["channels"][chnl])
    mip = np.clip(mip, 0, 200)  # clip for labels
    
    # add color labels to MIP
    text = []
    available = np.ones(mip.shape[0:2], dtype=bool)  # remaining unlabeled region
    masks = np.zeros(mip.shape[0:2], dtype=bool)
    for label in range(len(info)):
        key = (frame, label)
        if key in lookup:
            # read mask subset
            l, u = info[label,5:8].astype(int), info[label,8:11].astype(int)
            mask = read_h5(os.path.join(frame_path, "masks.h5"), to_fmt(label))

            # add color to nucleus (on remaining region)
            masks[l[1]:u[1],l[2]:u[2]] = np.max(mask, axis=0)
            mip[np.logical_and(masks, available)] += (55*lookup[key][1]).astype(np.uint8)
            available[masks] = 0
            masks *= False

            # append nucleus label
            x, y = int(info[label,3]), int(info[label,4])
            text.append([x, y, lookup[key][0]])
    mip = np.transpose(mip, axes=(1,0,2))  # transpose

    # add text labels to MIP
    h = mip.shape[0]
    mip = Image.fromarray(mip)  # convert for writing text

    draw = ImageDraw.Draw(mip)
    for i in range(len(text)):
        x, y, label = text[i]
        draw.text(xy=(x-20,y-15),text=to_fmt(label),
                  font=ImageFont.truetype(font="Arial", size=25), fill=(0,0,0))
    return mip


In [6]:
# inputs
base_path = "/mnt/md1/sumin/processed/MKH_dTAGProject/PolycombBodies/SUZ12/UNT/"
folder_path = "/mnt/md1/sumin/processed/MKH_dTAGProject/PolycombBodies/SUZ12/UNT/20260722_SUZ12_UNT_RING1B_HT_JFX650_SR2Y-01-processed"

nuclei_path = os.path.join(folder_path, 'nuclei') 
frame_path = os.path.join(folder_path, 'frames/000')


In [7]:
# make lookup for nuclei tracks

lookup = dict()
for item in os.listdir(nuclei_path):
    if item.isnumeric():                
        # get path to nucleus and read track
        nucleus_path = os.path.join(nuclei_path, to_fmt(int(item)))
        nucleus_track = np.loadtxt(os.path.join(nucleus_path, "info.csv"),
                                   delimiter=",", skiprows=1, ndmin=2)
        # get color and add
        color = make_color()
        for i in range(len(nucleus_track)):
            frame, label = int(nucleus_track[i,0]), int(nucleus_track[i,1])
            lookup[(frame, label)] = (int(item), color)

In [ ]:
# Check nucleus segmentation:

config = "airy_config.yaml"
c = read_yaml(config)
mip = labels_proc(0, frame_path, c, lookup)
mip

In [5]:
import os
from skimage.segmentation import mark_boundaries

def plot_nucleus_overlays(folder_path, output_base_folder):
    """
    For a single processed dataset folder, plot dcv MIP + body-label overlays
    for every nucleus, save one combined figure to output_base_folder.
    """
    nuclei_path = os.path.join(folder_path, 'nuclei')

    if not os.path.isdir(nuclei_path):
        print(f"Skipping {folder_path}: no 'nuclei' subfolder")
        return

    nuc_folders = sorted(
        os.path.join(nuclei_path, n) for n in os.listdir(nuclei_path)
        if os.path.isdir(os.path.join(nuclei_path, n))
    )

    valid_nuc_folders = []
    for nuc_folder in nuc_folders:
        body_file = os.path.join(nuc_folder, 'bodies', 'body_labels.h5')
        if os.path.exists(body_file):
            valid_nuc_folders.append(nuc_folder)
        else:
            print(f"Nucleus {os.path.basename(nuc_folder)}: no bodies file, skipping")

    if not valid_nuc_folders:
        print(f"Skipping {folder_path}: no valid nuclei found")
        return

    fig, ax = plt.subplots(nrows=len(valid_nuc_folders), ncols=4,
                            figsize=(12, 3 * len(valid_nuc_folders)))
    if len(valid_nuc_folders) == 1:
        ax = ax[np.newaxis, :]

    for row, nuc_folder in enumerate(valid_nuc_folders):
        nuc_id = os.path.basename(nuc_folder)
        lbl = nuc_id
        body_file = os.path.join(nuc_folder, 'bodies', 'body_labels.h5')

        mask = read_h5(os.path.join(folder_path, 'frames/000/masks.h5'), lbl)
        dcv  = read_h5(os.path.join(folder_path, 'frames/000/deconvolves.h5'), lbl)[0]
        body_labels = read_h5(body_file)

        plain_mip = to_mip(dcv, mask=mask, axis=0)
        ax[row, 0].imshow(plain_mip)
        ax[row, 0].axis('off')
        ax[row, 0].set_title(f'{nuc_id} (raw)', fontsize=9)

        for col, axis in zip([1, 2, 3], [0, 1, 2]):
            mip = to_mip(dcv, mask=mask, axis=axis)
            labels_mip = np.max(body_labels, axis=axis)
            overlay = mark_boundaries(mip.astype(float) / 255.0, labels_mip,
                                       color=(1, 0, 0), mode='outer')
            ax[row, col].imshow(overlay)
            ax[row, col].axis('off')
            ax[row, col].set_title(f'{nuc_id} (axis={axis})', fontsize=9)

    plt.tight_layout()

    os.makedirs(output_base_folder, exist_ok=True)
    dataset_name = os.path.basename(os.path.normpath(folder_path))
    out_path = os.path.join(output_base_folder, f"{dataset_name}_overlay.png")
    fig.savefig(out_path, dpi=150, bbox_inches='tight')
    plt.close(fig)

    print(f"Saved: {out_path}")



In [6]:

# --- loop over every dataset folder within base_path ---
conditions = ["SUZ12/UNT/", "SUZ12/dTAG/"]

base_path = '/mnt/md1/sumin/processed/MKH_dTAGProject/PolycombBodies'

for condition in conditions:
    input_folder = os.path.join(base_path, condition)
    output_folder = os.path.join(input_folder, 'Nucleus_overlay_outputs')

    dataset_folders = sorted(
        os.path.join(input_folder, d) for d in os.listdir(input_folder)
        if os.path.isdir(os.path.join(input_folder, d))
    )

    for folder_path in dataset_folders:
        plot_nucleus_overlays(folder_path, output_folder)


### DOWNLOAD THESE PNGs TO REVIEW AND MAKE A CSV FILE CALLED "nuclei_to_exclude.csv" under each conditions folder. 
# example entry for this csv:
# folder,nucleus
# 20260526_PCGF24_dTAG4h_RING1B_HT_JFX650_SR2Y-02-processed,008
# 20260526_PCGF24_dTAG4h_RING1B_HT_JFX650_SR2Y-04-processed,015

### RUN aggregate_body_info.ipynb to get the aggregate csv files of body counts/measurements, and exclude the nuclei with wrong segmentation
# The first two columns should match the folder,nucleus format perfectly.

Saved: /mnt/md1/sumin/processed/MKH_dTAGProject/PolycombBodies/SUZ12/UNT/Nucleus_overlay_outputs/20260722_SUZ12_UNT_RING1B_HT_JFX650_SR2Y-01-processed_overlay.png
Saved: /mnt/md1/sumin/processed/MKH_dTAGProject/PolycombBodies/SUZ12/UNT/Nucleus_overlay_outputs/20260722_SUZ12_UNT_RING1B_HT_JFX650_SR2Y-02-processed_overlay.png
Saved: /mnt/md1/sumin/processed/MKH_dTAGProject/PolycombBodies/SUZ12/UNT/Nucleus_overlay_outputs/20260722_SUZ12_UNT_RING1B_HT_JFX650_SR2Y-03-processed_overlay.png
Saved: /mnt/md1/sumin/processed/MKH_dTAGProject/PolycombBodies/SUZ12/UNT/Nucleus_overlay_outputs/20260722_SUZ12_UNT_RING1B_HT_JFX650_SR2Y-04-processed_overlay.png
Saved: /mnt/md1/sumin/processed/MKH_dTAGProject/PolycombBodies/SUZ12/UNT/Nucleus_overlay_outputs/20260722_SUZ12_UNT_RING1B_HT_JFX650_SR2Y-05-processed_overlay.png
Saved: /mnt/md1/sumin/processed/MKH_dTAGProject/PolycombBodies/SUZ12/UNT/Nucleus_overlay_outputs/20260722_SUZ12_UNT_RING1B_HT_JFX650_SR2Y-06-processed_overlay.png
Saved: /mnt/md1/sumin/